<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit

Audit the anonymized sample for useful signals, missingness, and measurement caveats before building a baseline.

Run the cells from top to bottom. The source dataset is the anonymized public starter sample; do not add private client data or credentials to this notebook.


## 1. Load and inspect

In [1]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
print("Column names:", df.columns.tolist())
display(df.dtypes.rename("dtype").to_frame())

Shape: (30000, 44)
Column names: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,dtype
content_id,str
client_id,str
search_volume,float64
competition,float64
competition_level,str
cpc,float64
content_type,str
main_intent,str
word_count,float64
char_count,float64


## 2. Missingness and target balance

In [2]:
missing = (df.isna().mean().mul(100).sort_values(ascending=False).rename("missing_pct").to_frame())
display(missing.head(15).round(2))
if "is_declining_label" in df:
    target = pd.to_numeric(df["is_declining_label"], errors="coerce")
elif "trend_direction" in df:
    target = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
else:
    raise KeyError("No recognized decline proxy column found.")
print("Observed decline-proxy rate:", round(float(target.mean()), 4))

,missing_pct
provider_used,71.46
word_count,25.66
char_count,25.66
word_count_tier,25.66
char_count_tier,25.66
model_used,19.11
trend_pct,11.29
competition_level,8.70
search_volume,8.23
cpc,8.23


Observed decline-proxy rate: 0.5421


## 3. Compare the proxy across interpretable groups

In [3]:
for col in ["position_tier", "staleness_bucket", "ga4_data_available"]:
    if col not in df.columns:
        continue
    temp = df[[col]].copy()
    temp["decline_proxy"] = target
    summary = temp.groupby(col, dropna=False)["decline_proxy"].agg(rows="size", decline_rate="mean").sort_values("rows", ascending=False)
    print("\\nGroup:", col)
    display(summary.round(3))

\nGroup: position_tier


,rows,decline_rate
position_tier,,
page_1,11814,0.570
striking,7304,0.610
page_3_5,7242,0.562
top_3,2321,0.241
deep,1319,0.344


## 4. Write an evidence-based audit summary

In [4]:
numeric = df.select_dtypes(include="number")
if len(numeric.columns) >= 2:
    corr = numeric.corr(numeric_only=True)
    if "is_declining_label" in corr.columns:
        signal = corr["is_declining_label"].drop("is_declining_label", errors="ignore").sort_values(key=lambda s: s.abs(), ascending=False)
        display(signal.head(10).to_frame("correlation_with_label").round(3))
    else:
        print("No numeric is_declining_label column; do not interpret correlations as causal evidence.")
print("Audit complete: check missingness, subgroup size, label construction, and whether each field is available at decision time.")

No numeric is_declining_label column; do not interpret correlations as causal evidence.
Audit complete: check missingness, subgroup size, label construction, and whether each field is available at decision time.
